In [1]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
import os
import kagglehub
import json

nltk.download('stopwords')
stop_words = set(stopwords.words("english"))
port_stem = PorterStemmer()

C:\Users\MAYUR\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\MAYUR\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [2]:
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'[^a-z ]', '', text)
    words = text.split()
    return " ".join([port_stem.stem(w) for w in words if w not in stop_words])

In [3]:
print("Downloading ISOT dataset via kagglehub...")
path = kagglehub.dataset_download("clmentbisaillon/fake-and-real-news-dataset")
print("Downloaded to:", path)

true_path = os.path.join(path, "True.csv")
fake_path = os.path.join(path, "Fake.csv")

df_true = pd.read_csv(true_path)
df_fake = pd.read_csv(fake_path)

df_true['label'] = 0 # 0 for True/Real
df_fake['label'] = 1 # 1 for Fake

df = pd.concat([df_true, df_fake], ignore_index=True)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
df.head()

Downloaded to: C:\Users\MAYUR\.cache\kagglehub\datasets\clmentbisaillon\fake-and-real-news-dataset\versions\1


,title,text,subject,date,label
0,BREAKING: GOP Chairman Grassley Has Had Enoug...,"Donald Trump s White House is in chaos, and th...",News,"July 21, 2017",1
1,Failed GOP Candidates Remembered In Hilarious...,Now that Donald Trump is the presumptive GOP n...,News,"May 7, 2016",1
2,Mike Pence’s New DC Neighbors Are HILARIOUSLY...,Mike Pence is a huge homophobe. He supports ex...,News,"December 3, 2016",1
3,California AG pledges to defend birth control ...,SAN FRANCISCO (Reuters) - California Attorney ...,politicsNews,"October 6, 2017",0
4,AZ RANCHERS Living On US-Mexico Border Destroy...,Twisted reasoning is all that comes from Pelos...,politics,"Apr 25, 2017",1


In [4]:
print(f"Applying clean_text to {len(df)} rows. This may take a few minutes...")
df['clean_text'] = df['text'].apply(clean_text)
df.head()

Applying clean_text to 44898 rows. This may take a few minutes...


,title,text,subject,date,label,clean_text
0,BREAKING: GOP Chairman Grassley Has Had Enoug...,"Donald Trump s White House is in chaos, and th...",News,"July 21, 2017",1,donald trump white hous chao tri cover russia ...
1,Failed GOP Candidates Remembered In Hilarious...,Now that Donald Trump is the presumptive GOP n...,News,"May 7, 2016",1,donald trump presumpt gop nomine time rememb c...
2,Mike Pence’s New DC Neighbors Are HILARIOUSLY...,Mike Pence is a huge homophobe. He supports ex...,News,"December 3, 2016",1,mike penc huge homophob support exgay convers ...
3,California AG pledges to defend birth control ...,SAN FRANCISCO (Reuters) - California Attorney ...,politicsNews,"October 6, 2017",0,san francisco reuter california attorney gener...
4,AZ RANCHERS Living On US-Mexico Border Destroy...,Twisted reasoning is all that comes from Pelos...,politics,"Apr 25, 2017",1,twist reason come pelosi day especi promin dem...


In [5]:
output_path = "processed_large_data.csv"
df.to_csv(output_path, index=False)
print("Saved to ", output_path)

Saved to  processed_large_data.csv


In [6]:
# Now update Fake_News.ipynb
notebook_path = "Fake_News.ipynb"
if os.path.exists(notebook_path):
    with open(notebook_path, "r", encoding="utf-8") as f:
        notebook = json.load(f)

    for cell in notebook.get('cells', []):
        if cell['cell_type'] == 'code':
            source = cell['source']
            for i, line in enumerate(source):
                if "pd.read_csv('fake.csv')" in line:
                    source[i] = line.replace("pd.read_csv('fake.csv')", "pd.read_csv('processed_large_data.csv')")
                if "pd.read_csv(\"fake.csv\")" in line:
                    source[i] = line.replace("pd.read_csv(\"fake.csv\")", "pd.read_csv('processed_large_data.csv')")
                if "data['text'].apply(clean_text)" in line:
                    source[i] = "# " + line.lstrip()

    with open(notebook_path, "w", encoding="utf-8") as f:
        json.dump(notebook, f, indent=1)
    print("Updated Fake_News.ipynb successfully.")

Updated Fake_News.ipynb successfully.
